In [1]:
import numpy as np
import torch
from torch import nn
import pyspiel

from poker_rnad_py import ActorThread

In [2]:
game_def = """universal_poker(
    betting=nolimit,
    bettingAbstraction=fullgame,
    numPlayers=6,
    blind=2 1 0 0 0 0,
    numRounds=4,
    firstPlayer=2 1 1 1,
    numSuits=4,
    numRanks=13,
    numHoleCards=2,
    numBoardCards=0 3 1 1,
    stack=200 200 200 200 200 200
)
""".replace("    ", "").replace("\n", "")
game_def

'universal_poker(betting=nolimit,bettingAbstraction=fullgame,numPlayers=6,blind=2 1 0 0 0 0,numRounds=4,firstPlayer=2 1 1 1,numSuits=4,numRanks=13,numHoleCards=2,numBoardCards=0 3 1 1,stack=200 200 200 200 200 200)'

In [3]:
class ResNet(nn.Module):
    def __init__(
            self, embedding_dim, dropout=0.0, prenorm=True, activation=nn.ReLU()):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            activation,
            nn.Dropout(dropout),
        )
        self.layernorm = nn.LayerNorm(embedding_dim)
        self.prenorm = prenorm
        nn.init.trunc_normal_(self.layer[0].weight, std=0.02, a=-0.04, b=0.04)

    def forward(self, x):
        if self.prenorm:
            return x + self.layer(self.layernorm(x))

        return self.layernorm(x + self.layer(x))


class RNadModel(nn.Module):
    def __init__(self, infostate_tensor_shape, num_actions, hidden_dim, dropout):
        super().__init__()

        self.tower = nn.Sequential(
            nn.Linear(infostate_tensor_shape, hidden_dim),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
            ResNet(hidden_dim, dropout),
        )
        self.policy_tower = nn.Linear(hidden_dim, num_actions)

        self.value_head = nn.Linear(hidden_dim, 1)
        self.log_policy_head = nn.Sequential(
            self.policy_tower,
            nn.LogSoftmax(dim=-1)
        )
        self.policy_head = nn.Sequential(
            self.policy_tower,
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        embedding = self.tower(x)
        return (
            self.value_head(embedding),
            self.log_policy_head(embedding),
            self.policy_head(embedding)
        )


class RNad:
    def __init__(self, game_def):
        self.game = pyspiel.load_game(game_def)
        infostate_tensor_shape = self.game.information_state_tensor_shape()[0]
        num_actions = self.game.num_distinct_actions()

        self.device = torch.device("cpu")
        self.model = RNadModel(
            infostate_tensor_shape=infostate_tensor_shape,
            num_actions=num_actions,
            hidden_dim=1024,
            dropout=0.1
        )
        print(sum(param.numel() for param in self.model.parameters()))
        jit_model = torch.jit.script(self.model).eval().to(self.device)
        self.actor = ActorThread(self.game, jit_model._c, 0)

In [ ]:
rnad = RNad(game_def)
traj = rnad.actor.generate_trajectories_batch(2 ** 17)
traj

7243978


[Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory(1 states),
 Trajectory